# 02 — Parear CEP × logradouro (Correios × Geosanja)

**Objetivo:** transferir o CEP da lista dos Correios para os segmentos de rua do Geosanja, cruzando pelo **nome do logradouro**.

**Como o cruzamento funciona (cascata):** cada nível só trabalha com o que sobrou do anterior.

| Nível | Chave | Tipo |
|---|---|---|
| 1 | tipo + nome + bairro/loteamento | exato |
| 2 | tipo + nome (só se o nome tiver **um único CEP** no Correios) | exato |
| 3 | nome (sem o tipo) + bairro/loteamento | exato |
| 4 | nome com **iniciais** (`Jose W de Carvalho` = `José Wagner de Carvalho`), mesmo tipo | palavra a palavra |
| 5 | nome parecido, dentro do mesmo tipo (abaixo do score 93 só com bairro compatível) | fuzzy (`rapidfuzz`) |

**Resultados (cada segmento do Geosanja termina em um destes status):**
`pareado` (1 CEP) · `multi_cep` (vários CEPs, mesma rua) · `ambiguo` (nome com vários CEPs e o bairro não decide) · `sem_cep` · `sem_nome`.

**Saídas** (pasta `03_processamento\02_parear_cep_logradouro`):
- `ponte_cep_logradouro.parquet` — tabela de ligação (um par CEP × segmento por linha)
- `logradouros_status_pareamento.gpkg` — resultado **automático** do pareamento (CEP e status por segmento). A base oficial editável é criada a partir dele na etapa 03
- `residuo_ceps_orfaos.csv`, `residuo_vias_sem_cep.csv`, `residuo_vias_sem_nome.csv`, `residuo_ambiguos.csv`, `residuo_multi_cep.csv`
- `auditoria_fuzzy.csv` — pareamentos fuzzy, dos menos aos mais parecidos, para você conferir o limiar

**Estrutura do notebook:** 1 Configuração · 2 Normalização · 3 Preparar Correios · 4 Preparar Geosanja · 5 Funções de pareamento · 6 Cascata · 7 Consolidação · 8 Resíduos e validação · 9 Exportação · 10 Resumo

## 1. Configuração
Caminhos, parâmetros e **dicionários editáveis**. Se quiser mudar uma regra (limiar do fuzzy, uma abreviação), é só aqui.

In [ ]:
import logging                       # registra mensagens com hora (melhor que print)
import re                             # expressões regulares (busca de padrões em texto)
import unicodedata                    # remover acentos
from pathlib import Path              # caminhos de arquivos
import numpy as np                    # contas com matrizes
import pandas as pd                   # tabelas
import pyogrio                        # leitura/escrita de arquivos geográficos
from rapidfuzz import fuzz, process   # comparação de textos "parecidos" (fuzzy)

# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "02_parear_cep_logradouro"          # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)

# --- Caminhos (config.yaml) ---------------------------------------------------
ARQ_CORREIOS = cfg.caminho("brutos.correios")       # exportação de logradouros dos correios (.xlsx)
ARQ_GEOSANJA = cfg.caminho("brutos.geosanja")       # malha de logradouros do geosanja (.gpkg)

# --- Parâmetros do pareamento ---------------------------------------------------
LIMIAR_BAIRRO = cfg.parametro("pareamento.limiar_bairro")       # similaridade mínima (0-100) para considerar que dois bairros/loteamentos são o mesmo
LIMIAR_FUZZY = cfg.parametro("pareamento.limiar_fuzzy")        # similaridade mínima (0-100) do nome no nível 4 (será calibrado com auditoria_fuzzy.csv)
LIMIAR_FUZZY_SEM_BAIRRO = cfg.parametro("pareamento.limiar_fuzzy_sem_bairro")   # fuzzy com score abaixo disso só vale se o bairro também for compatível
MIN_LEN_FUZZY = cfg.parametro("pareamento.min_len_fuzzy")        # nomes com menos caracteres que isso não entram no fuzzy (evita falsos positivos em nomes curtos)
SITUACOES_ACEITAS = {"Válido", "Aprovado"}           # situações do correios que entram no cruzamento

# --- Tipos de logradouro do correios (os de duas palavras vêm primeiro!) ------------
TIPOS_CORREIOS = ["AREA VERDE", "AREA RURAL", "COMUNIDADE URBANA", "RUA", "AVENIDA", "PRACA", "TRAVESSA",
                  "ALAMEDA", "ESTRADA", "VIELA", "RODOVIA", "COMUNIDADE", "AREA", "LARGO", "PONTE", "ACESSO",
                  "BULEVAR", "LADEIRA", "LOTEAMENTO", "MARGINAL", "PASSARELA", "ROTATORIA", "TREVO", "VIA", "VILA"]

# --- Tipos do geosanja que precisam ser convertidos para o equivalente do correios ---
TIPO_EQUIV_GEOSANJA = {"VIELA PUBLICA": "VIELA", "RUAV": "RUA", "RUA PROJETADA": "RUA"}   # edite se quiser

# --- Abreviações -> forma por extenso (aplicadas às DUAS bases) ------------------------
ABREVIACOES_TOKEN = {
    "DR": "DOUTOR", "DRA": "DOUTORA", "PROF": "PROFESSOR", "PROFA": "PROFESSORA", "ENG": "ENGENHEIRO",
    "ENGO": "ENGENHEIRO", "CEL": "CORONEL", "MAJ": "MAJOR", "CAP": "CAPITAO", "PE": "PADRE", "STO": "SANTO",
    "STA": "SANTA", "TV": "TRAVESSA", "AV": "AVENIDA", "PC": "PRACA", "PCA": "PRACA", "EST": "ESTRADA",
    "ROD": "RODOVIA", "SGT": "SARGENTO", "TEN": "TENENTE", "GEN": "GENERAL", "MONS": "MONSENHOR",
    "DEP": "DEPUTADO", "GOV": "GOVERNADOR", "PRES": "PRESIDENTE", "ALM": "ALMIRANTE", "MAL": "MARECHAL",
    # NÃO incluídos de propósito (significado incerto): VE, ME, CER, JOM -> ficam como estão e o fuzzy trata.
}

# --- Títulos que, quando aparecem DEPOIS da vírgula ("Fulano, Doutor"), vão para o início ---
TITULOS_POS_VIRGULA = {
    "DOUTOR", "DOUTORA", "DR", "DRA", "PROFESSOR", "PROFESSORA", "PROF", "PROFA", "ENGENHEIRO", "ENGENHEIRA",
    "ENG", "CORONEL", "CEL", "PADRE", "PE", "PASTOR", "MAJOR", "CAPITAO", "TENENTE", "SARGENTO", "GENERAL",
    "MONSENHOR", "DOM", "DONA", "SENADOR", "DEPUTADO", "VEREADOR", "GOVERNADOR", "PRESIDENTE", "MESTRE",
    "IRMA", "IRMAO", "FREI", "REVERENDO", "BISPO", "COMENDADOR", "CONEGO",
}

# Palavras genéricas de bairro: dois bairros que só diferem por elas são considerados iguais
# (ex.: 'Betania' = 'Vila Betania'). Vale só igualdade EXATA depois de retirá-las.
GENERICOS_BAIRRO = {"VILA", "JARDIM", "PARQUE", "RESIDENCIAL", "CONDOMINIO", "CONJUNTO", "HABITACIONAL",
                    "LOTEAMENTO", "CHACARA", "CHACARAS", "SITIO"}

PREPOSICOES = {"DE", "DA", "DO", "DAS", "DOS"}       # removidas dos nomes (o "E" NÃO é removido)

# --- Numerais por extenso -> dígitos (até 999 e combinações com "mil") ------------------
NUMERAIS = {
    "ZERO": 0, "UM": 1, "DOIS": 2, "TRES": 3, "QUATRO": 4, "CINCO": 5, "SEIS": 6, "SETE": 7, "OITO": 8, "NOVE": 9,
    "DEZ": 10, "ONZE": 11, "DOZE": 12, "TREZE": 13, "CATORZE": 14, "QUATORZE": 14, "QUINZE": 15, "DEZESSEIS": 16,
    "DEZASSEIS": 16, "DEZESSETE": 17, "DEZASSETE": 17, "DEZOITO": 18, "DEZENOVE": 19, "DEZANOVE": 19,
    "VINTE": 20, "TRINTA": 30, "QUARENTA": 40, "CINQUENTA": 50, "SESSENTA": 60, "SETENTA": 70, "OITENTA": 80,
    "NOVENTA": 90, "CEM": 100, "CENTO": 100, "DUZENTOS": 200, "TREZENTOS": 300, "QUATROCENTOS": 400,
    "QUINHENTOS": 500, "SEISCENTOS": 600, "SETECENTOS": 700, "OITOCENTOS": 800, "NOVECENTOS": 900,
}

# Regex que reconhece o tipo no início do logradouro do correios (já em maiúsculas e sem acento)
PADRAO_TIPO = r"^(" + "|".join(TIPOS_CORREIOS) + r")(?:\s+(.*))?$"

log.info("Configuração carregada!")
for _nome, _arq in [("correios", ARQ_CORREIOS), ("geosanja", ARQ_GEOSANJA)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())
log.info("Pasta de saída: %s", PASTA_SAIDA)

## 2. Normalização dos textos
A mesma função é aplicada ao **nome da rua, ao tipo e ao bairro** das duas bases. Passos, na ordem:

1. maiúsculas e sem acento → 2. remove texto entre parênteses (apelidos) → 3. `"Fulano, Doutor"` vira `"Doutor Fulano"` →
4. tira pontuação → 5. expande abreviações (`DR` → `DOUTOR`) → 6. numerais por extenso viram dígitos (`Vinte e Um` → `21`) →
7. remove `DE`, `DA`, `DO`, `DAS`, `DOS` → 8. junta letra+número (`H 27E` → `H27E`).

O trabalho pesado roda só nos **valores distintos** (muito mais rápido do que linha a linha).

In [ ]:
def texto_basico(serie: pd.Series) -> pd.Series:
    """Só MAIÚSCULAS, sem acento e sem espaços nas pontas (mantém a pontuação)."""
    s = serie.fillna("").astype(str)
    s = s.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")   # tira acentos
    return s.str.upper().str.strip()


def _numerais_para_digitos(texto: str) -> str:
    """Troca sequências de números por extenso por dígitos. Ex.: 'VINTE E UM' -> '21'; 'DOIS MIL E UM' -> '2001'."""
    tokens = texto.split()
    n = len(tokens)
    saida, i = [], 0
    while i < n:
        if tokens[i] in NUMERAIS or tokens[i] == "MIL":          # começou um número por extenso
            total, atual, j, ultimo_mil = 0, 0, i, False
            while True:
                t = tokens[j]
                if t == "MIL":                                    # 'mil' multiplica o que veio antes (ou vale 1000)
                    total += (atual or 1) * 1000
                    atual, ultimo_mil = 0, True
                else:
                    atual += NUMERAIS[t]
                    ultimo_mil = False
                j += 1
                if j >= n:
                    break
                if tokens[j] == "E" and j + 1 < n and (tokens[j + 1] in NUMERAIS or tokens[j + 1] == "MIL"):
                    j += 1                                        # 'e' ligando dois números ('vinte E um')
                    continue
                if tokens[j] == "MIL" and not ultimo_mil:         # 'dois MIL'
                    continue
                if ultimo_mil and tokens[j] in NUMERAIS:          # 'dois mil TREZENTOS'
                    continue
                break
            saida.append(str(total + atual))
            i = j
        else:
            saida.append(tokens[i])
            i += 1
    return " ".join(saida)


def _normalizar_um(txt: str, remover_parenteses: bool, converter_numerais: bool) -> str:
    """Aplica todos os passos de normalização a UM texto."""
    t = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("ascii").upper()   # 1) maiúsculas, sem acento
    if remover_parenteses:
        t = re.sub(r"\([^)]*\)", " ", t)                          # 2) tira '(Apelido)'
    t = re.sub(r"\s+", " ", t).strip()
    m = re.match(r"^(.*),\s*([A-Z]+)\.?\s*$", t)                  # 3) 'Fulano, Doutor' -> 'Doutor Fulano'
    if m and m.group(2) in TITULOS_POS_VIRGULA:
        t = f"{m.group(2)} {m.group(1)}"
    t = re.sub(r"[^A-Z0-9 ]", " ", t)                             # 4) pontuação vira espaço
    t = re.sub(r"\s+", " ", t).strip()
    t = " ".join(ABREVIACOES_TOKEN.get(x, x) for x in t.split())  # 5) abreviações por extenso
    if converter_numerais:
        t = _numerais_para_digitos(t)                             # 6) numerais -> dígitos
    t = " ".join(x for x in t.split() if x not in PREPOSICOES)    # 7) tira DE/DA/DO/DAS/DOS
    t = re.sub(r"\b([A-Z]{1,2}) (\d)", r"\1\2", t)                # 8) 'H 27E' -> 'H27E'
    t = re.sub(r"(\d) ([A-Z])\b", r"\1\2", t)                     #    '2 U'   -> '2U'
    return t


def normalizar_nome(serie: pd.Series, remover_parenteses: bool = True, converter_numerais: bool = True) -> pd.Series:
    """Normaliza uma coluna de texto (roda só nos valores distintos e espalha o resultado)."""
    s = serie.fillna("").astype(str)
    mapa = {u: _normalizar_um(u, remover_parenteses, converter_numerais) for u in s.unique()}
    return s.map(mapa)


def assinatura_digitos(serie: pd.Series) -> pd.Series:
    """Junta todos os números do texto (ex.: 'H10B 7' -> '107'). Usada para só comparar nomes com os mesmos números."""
    return serie.str.findall(r"\d+").str.join("")


def limpar_texto(serie: pd.Series) -> pd.Series:
    """Tira espaços das pontas e transforma vazio em nulo."""
    s = serie.astype("string").str.strip()
    return s.mask(s == "", pd.NA)


def validar_tabela(nome, df, chave=None, obrigatorias=None, linhas_entrada=None):
    """Bloco padrão de validação: linhas, perda vs. entrada, nulos/vazios nas colunas obrigatórias e duplicados na chave."""
    log.info("Validando %s...", nome)
    log.info("  linhas: %d", len(df))
    if linhas_entrada is not None:
        perda = linhas_entrada - len(df)
        log.info("  perda vs. entrada: %d (%.2f%%)", perda, (perda / linhas_entrada * 100) if linhas_entrada else 0.0)
    for col in (obrigatorias or []):
        vazios = int((df[col].isna() | (df[col].astype(str).str.strip() == "")).sum())
        (log.info if vazios == 0 else log.warning)("  '%s': %d nulos/vazios", col, vazios)
    if chave:
        dup = int(df.duplicated(chave).sum())
        (log.info if dup == 0 else log.warning)("  duplicados na chave %s: %d", chave, dup)
    log.info("Validação de %s concluída!", nome)


log.info("Funções de normalização definidas!")

### Teste da normalização
Mostra, para exemplos reais encontrados nas duas bases, como cada texto fica depois da normalização.
Confira se o resultado faz sentido antes de seguir.

In [ ]:
log.info("Testando normalização...")
_exemplos = pd.Series([
    "Marco Antonio Sala, Doutor", "Luiz Fernando Leite Chaves, Dr", "Djalma Sebastiao Henze Aalves (Djalma Carioca)",
    "Vinte e Um de Abril", "Cento e Quatorze", "Ub Dois Mil e Um", "Alcides de M. Paula", "Joao V Pelegrino",
    "H 27E", "H10B", "2 U", "Rua 15 de Novembro", "Quinze de Novembro", "1ª Tv. Mato Dentro", "Doutor Nelson D'Avila",
    "Marechal-do-Ar Eduardo Gomes", "Beija-Flor", "Estrada Municipal SJC-145", "Sessenta e Um",
])
_teste = pd.DataFrame({"original": _exemplos, "normalizado": normalizar_nome(_exemplos)})
log.info("Resultado:\n%s", _teste.to_string())
log.info("Teste de normalização concluído!")

## 3. Preparar a base dos Correios
- Lê a planilha (tudo como texto) e **mantém as situações `Válido` e `Aprovado`**.
- CEP só com dígitos, 8 posições.
- Separa o **tipo** (`Rua`, `Avenida`, `Área Rural`...) do **nome**.
- Gera as chaves normalizadas de nome, bairro e complemento (`Conj Habitac/Loteam`).

In [ ]:
def preparar_correios(caminho: Path) -> pd.DataFrame:
    """Lê e padroniza a planilha dos correios. Devolve uma linha por linha da planilha aceita."""
    log.info("Lendo planilha dos correios...")
    bruto = pd.read_excel(caminho, sheet_name="Page 1", dtype=str, keep_default_na=False)
    log.info("Planilha lida: %d linhas!", len(bruto))

    log.info("Padronizando correios...")
    df = bruto[bruto["Situação"].isin(SITUACOES_ACEITAS)].copy()           # filtra situação
    log.info("  linhas fora das situações aceitas: %d", len(bruto) - len(df))

    out = pd.DataFrame({
        "cep": df["CEP"].str.replace(r"\D", "", regex=True).str.zfill(8),  # 12345-678 -> 12345678
        "logradouro_orig": df["Logradouro"].str.strip(),
        "bairro_orig": df["Bairro"].str.strip(),
        "conj_orig": df["Conj Habitac/Loteam/Inf. Adicional"].str.strip(),
        "tipo_codificacao": df["Tipo Codificação"].str.strip(),
    }).reset_index(drop=True)

    base = texto_basico(out["logradouro_orig"])                            # texto simples para achar o tipo
    ext = base.str.extract(PADRAO_TIPO)                                    # coluna 0 = tipo, coluna 1 = resto (nome)
    out["tipo_key"] = ext[0].fillna("")                                    # sem tipo reconhecido -> vazio
    nome_cru = ext[1].where(ext[0].notna(), base).fillna("")               # sem tipo: o nome é o texto inteiro
    out["nome_key"] = normalizar_nome(nome_cru)
    out["bairro_key"] = normalizar_nome(out["bairro_orig"], remover_parenteses=False)
    out["conj_key"] = normalizar_nome(out["conj_orig"], remover_parenteses=False)   # aqui o texto entre () é útil
    log.info("correios padronizado: %d linhas, %d CEPs distintos!", len(out), out["cep"].nunique())
    validar_tabela("correios padronizado", out, chave=["cep", "logradouro_orig", "bairro_orig"],
                   obrigatorias=["cep", "logradouro_orig", "bairro_orig"], linhas_entrada=len(bruto))
    log.info("Linhas sem tipo reconhecido: %d | com nome vazio (ex.: 'Área Rural'): %d",
             int((out["tipo_key"] == "").sum()), int((out["nome_key"] == "").sum()))
    return out


correios = preparar_correios(ARQ_CORREIOS)
log.info("Amostra:\n%s", correios[["cep", "logradouro_orig", "tipo_key", "nome_key", "bairro_key"]].sample(8, random_state=1).to_string())

## 4. Preparar a base do Geosanja
- Lê só a tabela de atributos (`id`, `tipo`, `nome`, `loteamento`).
- Marca as vias **sem nome** (nulas ou `Sem nome`): não entram no pareamento.
- Monta as **unidades de pareamento**: combinações distintas de (tipo, nome, loteamento). Cada unidade é pareada uma vez e depois o resultado volta para todos os segmentos dela.

In [ ]:
def preparar_geosanja(caminho: Path):
    """Lê e padroniza os atributos do geosanja. Devolve (segmentos, unidades_de_pareamento, nome_da_camada)."""
    log.info("Lendo atributos do geosanja...")
    camada = str(pyogrio.list_layers(caminho)[0][0])
    bruto = pyogrio.read_dataframe(caminho, layer=camada, read_geometry=False, columns=["id", "tipo", "nome", "loteamento"])
    log.info("Atributos lidos: %d feições (camada '%s')!", len(bruto), camada)

    log.info("Padronizando geosanja...")
    seg = pd.DataFrame({
        "id": bruto["id"],
        "tipo_orig": limpar_texto(bruto["tipo"]),
        "nome_orig": limpar_texto(bruto["nome"]),
        "lote_orig": limpar_texto(bruto["loteamento"]),
    })
    seg["tipo_key"] = normalizar_nome(seg["tipo_orig"], False, False).replace(TIPO_EQUIV_GEOSANJA)
    seg["nome_key"] = normalizar_nome(seg["nome_orig"])
    seg["lote_key"] = normalizar_nome(seg["lote_orig"], remover_parenteses=False)
    # vias sem nome: vazio, 'Sem nome' ou 'Sem nome 70' (com número) -> não são pareadas
    seg["sem_nome"] = seg["nome_key"].isin([""]) | seg["nome_key"].str.match(r"^SEM NOME\b")

    unid = (seg.loc[~seg["sem_nome"], ["tipo_key", "nome_key", "lote_key"]]
            .drop_duplicates().reset_index(drop=True))
    unid["uid"] = unid.index                                               # identificador da unidade
    seg = seg.merge(unid, on=["tipo_key", "nome_key", "lote_key"], how="left")   # devolve o uid para cada segmento
    seg["uid"] = seg["uid"].astype("Int64")

    log.info("geosanja padronizado: %d segmentos | %d sem nome | %d unidades de pareamento!",
             len(seg), int(seg["sem_nome"].sum()), len(unid))
    validar_tabela("geosanja (segmentos)", seg, chave=["id"], obrigatorias=["id"], linhas_entrada=len(bruto))
    return seg, unid, camada


geo, unidades, CAMADA_GEO = preparar_geosanja(ARQ_GEOSANJA)
log.info("Amostra:\n%s", geo.loc[~geo["sem_nome"], ["id", "tipo_orig", "nome_orig", "tipo_key", "nome_key", "lote_key"]].sample(8, random_state=1).to_string())

## 5. Funções de pareamento
- `_com_score_bairro`: calcula quão parecido é o loteamento do Geosanja com o bairro (ou o complemento) do Correios.
- `_decidir`: depois de achar os CEPs candidatos de cada unidade, decide se **pareia**, se fica **ambígua**, ou nenhum.
  - modo `bairro`: só aceita candidatos com bairro compatível.
  - modo `unico`: só aceita se o nome tiver **um único CEP**; com vários, a unidade vira ambígua.
  - modo `geral`: usa o bairro para filtrar; sem bairro compatível, aceita só se houver um único CEP.
- `_nivel_iniciais`: compara palavra a palavra, aceitando uma inicial no lugar do nome completo.
- `_nivel_fuzzy`: compara nomes parecidos dentro do mesmo tipo e com os mesmos números.
- Bairros que só diferem por palavras genéricas (`Vila`, `Jardim`, `Condomínio`...) valem como iguais (igualdade exata).

In [ ]:
# Tabela do correios com colunas prefixadas por "c_" (para não confundir com as colunas do geosanja)
corr_c = (correios.loc[correios["nome_key"] != "", ["cep", "tipo_key", "nome_key", "bairro_key", "conj_key"]]
          .rename(columns=lambda c: "c_" + c))


def _com_score_bairro(cand: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta 'score_bairro': o melhor entre (loteamento × bairro) e (loteamento × complemento), de 0 a 100."""
    pares = cand[["lote_key", "c_bairro_key", "c_conj_key"]].drop_duplicates().copy()

    def _sem_genericos(texto):
        """Tira palavras genéricas de bairro (Vila, Jardim, Parque, Condomínio...)."""
        return " ".join(t for t in texto.split() if t not in GENERICOS_BAIRRO)

    def _um(lote, outro):
        if not outro:
            return 0.0
        base = _sem_genericos(lote)
        if base and base == _sem_genericos(outro):
            return 100.0                                      # só diferem por palavra genérica: tratados como iguais
        return fuzz.ratio(lote, outro)

    def _score(lote, bairro, conj):
        if not lote:
            return 0.0                                        # sem loteamento não há como comparar
        return max(_um(lote, bairro), _um(lote, conj))

    pares["score_bairro"] = [_score(l, b, c) for l, b, c in zip(pares["lote_key"], pares["c_bairro_key"], pares["c_conj_key"])]
    return cand.merge(pares, on=["lote_key", "c_bairro_key", "c_conj_key"], how="left")


def _vazios():
    """Tabelas vazias no formato de saída de _decidir."""
    return (pd.DataFrame(columns=["uid", "cep", "score_nome", "score_bairro"]),
            pd.DataFrame(columns=["uid", "ceps_candidatos", "n_candidatos"]))


def _decidir(cand: pd.DataFrame, modo: str, piso_sem_bairro: float = None):
    """Decide, para cada unidade, entre parear ou marcar como ambígua. Devolve (pareados, ambiguos).

    piso_sem_bairro: se informado, candidatos com score_nome abaixo dele só valem se o bairro também for compatível.
    """
    if cand.empty:
        return _vazios()
    cand = _com_score_bairro(cand)
    cand["bairro_ok"] = cand["score_bairro"] >= LIMIAR_BAIRRO
    if piso_sem_bairro is not None:
        cand = cand[(cand["score_nome"] >= piso_sem_bairro) | cand["bairro_ok"]]   # descarta fuzzy "fraco" sem bairro
        if cand.empty:
            return _vazios()
    por_cep = (cand.groupby(["uid", "c_cep"], as_index=False)
               .agg(score_nome=("score_nome", "max"), score_bairro=("score_bairro", "max"), bairro_ok=("bairro_ok", "max")))
    algum_ok = por_cep.groupby("uid")["bairro_ok"].transform("max")        # a unidade tem algum candidato com bairro compatível?
    n_ceps = por_cep.groupby("uid")["c_cep"].transform("nunique")          # quantos CEPs candidatos a unidade tem

    if modo == "bairro":
        sel_par, sel_amb = por_cep["bairro_ok"], pd.Series(False, index=por_cep.index)
    elif modo == "unico":
        sel_par, sel_amb = n_ceps == 1, n_ceps > 1
    else:  # "geral"
        sel_par = (algum_ok & por_cep["bairro_ok"]) | (~algum_ok & (n_ceps == 1))
        sel_amb = ~algum_ok & (n_ceps > 1)

    pareados = por_cep.loc[sel_par, ["uid", "c_cep", "score_nome", "score_bairro"]].rename(columns={"c_cep": "cep"})
    amb = por_cep.loc[sel_amb]
    ambiguos = (amb.groupby("uid").agg(ceps_candidatos=("c_cep", lambda s: "|".join(sorted(set(s)))),
                                       n_candidatos=("c_cep", "nunique")).reset_index())
    return pareados, ambiguos


def _iniciais_compativeis(a: str, b: str) -> bool:
    """True se os dois nomes têm o mesmo nº de palavras, cada palavra igual ou uma inicial da outra, e há ao menos 1 inicial."""
    ta, tb = a.split(), b.split()
    if len(ta) != len(tb):
        return False
    achou_inicial = False
    for x, y in zip(ta, tb):
        if x == y:
            continue
        if len(x) == 1 and x.isalpha() and y.startswith(x):        # 'W' casa com 'WAGNER'
            achou_inicial = True
        elif len(y) == 1 and y.isalpha() and x.startswith(y):
            achou_inicial = True
        else:
            return False
    return achou_inicial


def _nivel_iniciais(restantes: pd.DataFrame) -> pd.DataFrame:
    """Nível 4: candidatos por nome com iniciais, dentro do mesmo tipo. Só compara nomes com o mesmo nº de palavras
    e a mesma letra inicial na primeira e na última palavra (reduz muito o número de comparações)."""
    def _prep(df, col):
        out = df.copy()
        palavras = out[col].str.split()
        out["ntok"] = palavras.str.len()
        out["f1"] = out[col].str[:1]
        out["fl"] = palavras.str[-1].str[:1]
        return out

    q = _prep(restantes, "nome_key")
    ch = _prep(corr_c[["c_tipo_key", "c_nome_key"]].drop_duplicates().reset_index(drop=True), "c_nome_key")
    achados = []
    for (tipo, ntok, f1, fl), g in q[q["ntok"] >= 2].groupby(["tipo_key", "ntok", "f1", "fl"]):
        mascara = (ch["ntok"] == ntok) & (ch["f1"] == f1) & (ch["fl"] == fl)
        if tipo:
            mascara &= ch["c_tipo_key"] == tipo                    # sem tipo no geosanja: compara com todos os tipos
        escolhas = ch[mascara]
        if escolhas.empty:
            continue
        for nome in g["nome_key"].drop_duplicates():
            for c_tipo, c_nome in zip(escolhas["c_tipo_key"], escolhas["c_nome_key"]):
                if _iniciais_compativeis(nome, c_nome):
                    achados.append((tipo, nome, c_tipo, c_nome))
    if not achados:
        return pd.DataFrame()
    pares = pd.DataFrame(achados, columns=["tipo_key", "nome_key", "c_tipo_key", "c_nome_key"])
    cand = restantes.merge(pares, on=["tipo_key", "nome_key"], how="inner")
    cand = cand.merge(corr_c, on=["c_tipo_key", "c_nome_key"], how="inner")
    cand["score_nome"] = 100.0
    return cand


def _nivel_fuzzy(restantes: pd.DataFrame) -> pd.DataFrame:
    """Nível 4: candidatos por nome parecido (token_sort_ratio >= LIMIAR_FUZZY), dentro do mesmo tipo e com os mesmos números."""
    q = restantes[restantes["nome_key"].str.len() >= MIN_LEN_FUZZY].copy()
    q["sig"] = assinatura_digitos(q["nome_key"])
    ch = corr_c[["c_tipo_key", "c_nome_key"]].drop_duplicates().reset_index(drop=True)
    ch["sig"] = assinatura_digitos(ch["c_nome_key"])

    achados = []                                                           # lista de pares (nome do geosanja, nome do correios, score)
    for (tipo, sig), g in q.groupby(["tipo_key", "sig"]):                  # blocos: mesmo tipo e mesmos números
        mascara = ch["sig"] == sig
        if tipo:
            mascara &= ch["c_tipo_key"] == tipo                            # sem tipo no geosanja: compara com todos os tipos
        escolhas = ch[mascara]
        if escolhas.empty:
            continue
        nomes_q = g["nome_key"].drop_duplicates().tolist()
        nomes_c = escolhas["c_nome_key"].tolist()
        matriz = process.cdist(nomes_q, nomes_c, scorer=fuzz.token_sort_ratio, dtype=np.float32,
                               score_cutoff=LIMIAR_FUZZY, workers=-1)      # matriz de similaridades (abaixo do corte = 0)
        melhores = matriz.max(axis=1)
        for i, nome in enumerate(nomes_q):
            if melhores[i] < LIMIAR_FUZZY:
                continue
            for k in np.flatnonzero(matriz[i] >= melhores[i] - 1e-6):      # todos os empatados no melhor score
                achados.append((tipo, nome, escolhas["c_tipo_key"].iloc[k], escolhas["c_nome_key"].iloc[k], float(melhores[i])))
    if not achados:
        return pd.DataFrame(columns=list(restantes.columns) + ["c_cep", "c_bairro_key", "c_conj_key", "score_nome"])
    pares = pd.DataFrame(achados, columns=["tipo_key", "nome_key", "c_tipo_key", "c_nome_key", "score_nome"])
    cand = restantes.merge(pares, on=["tipo_key", "nome_key"], how="inner")            # volta para as unidades
    return cand.merge(corr_c, on=["c_tipo_key", "c_nome_key"], how="inner")            # traz os CEPs dos nomes achados


log.info("Funções de pareamento definidas!")

## 6. Cascata de pareamento
Cada nível registra no log: unidades que entraram, pareadas, ambíguas e as que seguem para o próximo nível.

In [ ]:
def executar_cascata(unid: pd.DataFrame):
    """Roda os 5 níveis. Devolve (pareados, ambiguos, restantes) no nível de unidade."""
    restantes = unid.copy()
    todos_pareados, todos_ambiguos = [], []

    def _fechar(nivel, metodo, pareados, ambiguos, n_entrada):
        """Registra o resultado do nível e retira das 'restantes' o que foi resolvido."""
        nonlocal restantes
        pareados = pareados.assign(nivel=nivel, metodo=metodo)
        ambiguos = ambiguos.assign(nivel=nivel)
        todos_pareados.append(pareados)
        todos_ambiguos.append(ambiguos)
        resolvidas = set(pareados["uid"]) | set(ambiguos["uid"])
        restantes = restantes[~restantes["uid"].isin(resolvidas)]
        log.info("Nível %d (%s): entraram %d | pareadas %d | ambíguas %d | seguem %d!",
                 nivel, metodo, n_entrada, pareados["uid"].nunique(), ambiguos["uid"].nunique(), len(restantes))

    # ---- Nível 1: tipo + nome + bairro (exato) ----
    log.info("Nível 1: tipo + nome + bairro...")
    n = len(restantes)
    cand = restantes.merge(corr_c, left_on=["tipo_key", "nome_key"], right_on=["c_tipo_key", "c_nome_key"], how="inner")
    cand["score_nome"] = 100.0
    par, amb = _decidir(cand, "bairro")
    _fechar(1, "exato_tipo_nome_bairro", par, amb, n)

    # ---- Nível 2: tipo + nome, nome com CEP único (exato) ----
    log.info("Nível 2: tipo + nome (CEP único)...")
    n = len(restantes)
    cand = restantes.merge(corr_c, left_on=["tipo_key", "nome_key"], right_on=["c_tipo_key", "c_nome_key"], how="inner")
    cand["score_nome"] = 100.0
    par, amb = _decidir(cand, "unico")
    _fechar(2, "exato_tipo_nome_unico", par, amb, n)

    # ---- Nível 3: nome (sem tipo) + bairro (exato) ----
    log.info("Nível 3: nome + bairro (sem o tipo)...")
    n = len(restantes)
    cand = restantes.merge(corr_c, left_on="nome_key", right_on="c_nome_key", how="inner")
    cand["score_nome"] = 100.0
    par, amb = _decidir(cand, "bairro")
    _fechar(3, "exato_nome_bairro", par, amb, n)

    # ---- Nível 4: iniciais ----
    log.info("Nível 4: nomes com iniciais...")
    n = len(restantes)
    cand = _nivel_iniciais(restantes)
    par, amb = _decidir(cand, "geral")
    _fechar(4, "iniciais", par, amb, n)

    # ---- Nível 5: fuzzy ----
    log.info("Nível 5: fuzzy (limiar %d; abaixo de %d só com bairro compatível)...", LIMIAR_FUZZY, LIMIAR_FUZZY_SEM_BAIRRO)
    n = len(restantes)
    cand = _nivel_fuzzy(restantes)
    par, amb = _decidir(cand, "geral", piso_sem_bairro=LIMIAR_FUZZY_SEM_BAIRRO)
    _fechar(5, "fuzzy", par, amb, n)

    return (pd.concat(todos_pareados, ignore_index=True),
            pd.concat(todos_ambiguos, ignore_index=True),
            restantes)


log.info("Iniciando cascata de pareamento...")
pareados_u, ambiguos_u, restantes_u = executar_cascata(unidades)
log.info("Cascata concluída!")
log.info("Unidades: %d | pareadas %d | ambíguas %d | sem CEP %d",
         len(unidades), pareados_u["uid"].nunique(), ambiguos_u["uid"].nunique(), len(restantes_u))
validar_tabela("Pareados (unidade × CEP)", pareados_u, chave=["uid", "cep"], obrigatorias=["uid", "cep"])
_dup_uid = set(pareados_u["uid"]) & set(ambiguos_u["uid"])
(log.info if not _dup_uid else log.warning)("Unidades ao mesmo tempo pareadas e ambíguas: %d", len(_dup_uid))

## 7. Consolidação
Devolve o resultado das unidades para **cada segmento** do Geosanja e monta:
- a **tabela de ligação** `ponte` (um par segmento × CEP por linha, com método, score e contagens);
- o **status** de cada segmento: `pareado`, `multi_cep`, `ambiguo`, `sem_cep` ou `sem_nome`.

In [ ]:
log.info("Consolidando resultados...")

# --- Informações do correios por CEP (para anexar na ponte) ----------------------------
info_cep = (correios.groupby("cep")
            .agg(logradouro_correios=("logradouro_orig", lambda s: " | ".join(dict.fromkeys(s))),   # grafias alternativas juntas
                 bairro_correios=("bairro_orig", "first"),
                 tipo_codificacao=("tipo_codificacao", "first"))
            .reset_index())

# --- Tabela de ligação: segmento × CEP -------------------------------------------------
ponte = (geo.loc[geo["uid"].notna(), ["id", "uid", "tipo_orig", "nome_orig", "lote_orig"]]
         .merge(pareados_u[["uid", "cep", "nivel", "metodo", "score_nome", "score_bairro"]], on="uid", how="inner")
         .merge(info_cep, on="cep", how="left")
         .drop(columns="uid")
         .rename(columns={"id": "id_geosanja"}))
ponte["n_ceps_via"] = ponte.groupby("id_geosanja")["cep"].transform("nunique")         # quantos CEPs o segmento recebeu
ponte["n_segmentos_cep"] = ponte.groupby("cep")["id_geosanja"].transform("nunique")    # quantos segmentos usam o CEP
ponte["multi_cep"] = ponte["n_ceps_via"] > 1
validar_tabela("Ponte CEP × logradouro", ponte, chave=["id_geosanja", "cep"], obrigatorias=["id_geosanja", "cep", "metodo"])

# --- Status por segmento -------------------------------------------------------------------
agg_p = (ponte.groupby("id_geosanja")
         .agg(n_ceps=("cep", "nunique"), ceps=("cep", lambda s: "|".join(sorted(set(s)))),
              metodo=("metodo", "first"), score_nome=("score_nome", "max"))
         .reset_index().rename(columns={"id_geosanja": "id"}))
amb_seg = ambiguos_u[["uid", "ceps_candidatos", "n_candidatos", "nivel"]].rename(
    columns={"ceps_candidatos": "amb_ceps", "n_candidatos": "amb_n", "nivel": "amb_nivel"})

status = (geo[["id", "uid", "tipo_orig", "nome_orig", "lote_orig", "sem_nome"]]
          .merge(agg_p, on="id", how="left")
          .merge(amb_seg, on="uid", how="left"))
status["n_ceps"] = status["n_ceps"].fillna(0).astype(int)
condicoes = [status["sem_nome"], status["n_ceps"] == 1, status["n_ceps"] > 1, status["amb_ceps"].notna()]
status["status"] = np.select(condicoes, ["sem_nome", "pareado", "multi_cep", "ambiguo"], default="sem_cep")
status["cep"] = status["ceps"].where(status["n_ceps"] == 1)                                   # CEP único
status["ceps_candidatos"] = status["ceps"].where(status["n_ceps"] > 1, status["amb_ceps"])    # vários CEPs (multi_cep ou ambíguo)
status["metodo"] = status["metodo"].where(status["n_ceps"] > 0, np.where(status["amb_ceps"].notna(), "ambiguo", None))
status = status[["id", "status", "cep", "ceps_candidatos", "n_ceps", "metodo", "score_nome",
                 "tipo_orig", "nome_orig", "lote_orig"]]

validar_tabela("Status por segmento", status, chave=["id"], obrigatorias=["id", "status"], linhas_entrada=len(geo))
log.info("Contagem por status:\n%s", status["status"].value_counts().to_string())
assert len(status) == len(geo), "Perda de segmentos na consolidação!"
log.info("Consolidação concluída!")

## 8. Resíduos e validação
Resíduos para auditoria:
- **CEPs órfãos:** CEPs dos Correios sem nenhum segmento pareado (a coluna `em_ambiguo` indica os que aparecem como candidatos em vias ambíguas)
- **Vias sem CEP:** segmentos nomeados do Geosanja sem pareamento nem ambiguidade
- **Vias sem nome:** segmentos `Sem nome`
- **Ambíguos:** segmentos com vários CEPs candidatos que o bairro não resolveu
- **Multi-CEP:** segmentos pareados com mais de um CEP (seccionamento)
- **Auditoria do fuzzy:** pareamentos do nível 4, do menor ao maior score

In [ ]:
log.info("Montando resíduos...")

# --- CEPs órfãos ------------------------------------------------------------------------------------
ceps_pareados = set(ponte["cep"])
ceps_em_ambiguos = set("|".join(ambiguos_u["ceps_candidatos"].dropna()).split("|")) - {""}
res_ceps_orfaos = (correios[~correios["cep"].isin(ceps_pareados)]
                   [["cep", "logradouro_orig", "tipo_key", "bairro_orig", "conj_orig", "tipo_codificacao"]]
                   .assign(em_ambiguo=lambda d: d["cep"].isin(ceps_em_ambiguos))
                   .sort_values(["tipo_key", "logradouro_orig"]).reset_index(drop=True))

# --- Vias do geosanja ----------------------------------------------------------------------------------
res_vias_sem_cep = status[status["status"] == "sem_cep"].drop(columns=["cep", "ceps_candidatos", "n_ceps", "metodo", "score_nome"])
res_vias_sem_nome = status[status["status"] == "sem_nome"][["id", "tipo_orig", "lote_orig"]]
res_ambiguos = status[status["status"] == "ambiguo"][["id", "tipo_orig", "nome_orig", "lote_orig", "ceps_candidatos"]]
res_multi_cep = (ponte[ponte["multi_cep"]]
                 [["id_geosanja", "nome_orig", "lote_orig", "cep", "logradouro_correios", "bairro_correios", "metodo", "n_ceps_via"]]
                 .sort_values(["id_geosanja", "cep"]))
aud_fuzzy = (ponte[ponte["metodo"] == "fuzzy"]
             [["nome_orig", "tipo_orig", "lote_orig", "logradouro_correios", "bairro_correios", "cep", "score_nome", "score_bairro"]]
             .drop_duplicates(["nome_orig", "tipo_orig", "lote_orig", "cep"])
             .sort_values("score_nome").reset_index(drop=True))

# --- Validação final: tudo precisa fechar a conta -------------------------------------------------------
n_seg = len(geo)
soma = int(status["status"].value_counts().sum())
(log.info if soma == n_seg else log.error)("[%s] Segmentos: %d no geosanja = %d nos status", "OK" if soma == n_seg else "ERRO", n_seg, soma)
n_ceps_total = correios["cep"].nunique()
fecha = len(ceps_pareados) + len(res_ceps_orfaos["cep"].unique()) == n_ceps_total
(log.info if fecha else log.error)("[%s] CEPs: %d pareados + %d órfãos = %d distintos nos correios",
                                   "OK" if fecha else "ERRO", len(ceps_pareados), res_ceps_orfaos["cep"].nunique(), n_ceps_total)

for nome, tab in [("CEPs órfãos", res_ceps_orfaos), ("Vias sem CEP", res_vias_sem_cep), ("Vias sem nome", res_vias_sem_nome),
                  ("Ambíguos", res_ambiguos), ("Multi-CEP", res_multi_cep), ("Auditoria fuzzy", aud_fuzzy)]:
    log.info("  resíduo '%s': %d linhas", nome, len(tab))

log.info("Órfãos por tipo (top 12):\n%s", res_ceps_orfaos["tipo_key"].replace("", "(sem tipo)").value_counts().head(12).to_string())
log.info("Vias sem CEP por tipo (top 12):\n%s", res_vias_sem_cep["tipo_orig"].fillna("(sem tipo)").value_counts().head(12).to_string())
log.info("Resíduos montados!")

## 9. Exportação
Salva a ponte em parquet, os resíduos em CSV (abre direto no Excel) e a camada de logradouros com CEP e status em GeoPackage.

In [ ]:
log.info("Exportando resultados...")

# --- Tabela de ligação (parquet) ----------------------------------------------------------------------------
arq_ponte = PASTA_SAIDA / "ponte_cep_logradouro.parquet"
ponte.to_parquet(arq_ponte, index=False)
log.info("Ponte salva: %s (%d linhas)", arq_ponte.name, len(ponte))

# --- Resíduos e auditoria (CSV com separador ';' e acentos corretos no Excel) ---------------------------
for nome_arq, tab in {"residuo_ceps_orfaos": res_ceps_orfaos, "residuo_vias_sem_cep": res_vias_sem_cep,
                      "residuo_vias_sem_nome": res_vias_sem_nome, "residuo_ambiguos": res_ambiguos,
                      "residuo_multi_cep": res_multi_cep, "auditoria_fuzzy": aud_fuzzy}.items():
    tab.to_csv(PASTA_SAIDA / f"{nome_arq}.csv", sep=";", index=False, encoding="utf-8-sig")
    log.info("  %s.csv salvo (%d linhas)", nome_arq, len(tab))

# --- Camada de logradouros com CEP (GeoPackage) ---------------------------------------------------------------
log.info("Montando camada com geometria...")
gdf = pyogrio.read_dataframe(ARQ_GEOSANJA, layer=CAMADA_GEO, columns=["id"])       # só id + geometria
gdf = gdf.merge(status, on="id", how="left")
for col in ["status", "cep", "ceps_candidatos", "metodo", "tipo_orig", "nome_orig", "lote_orig"]:
    gdf[col] = gdf[col].astype(object).where(gdf[col].notna(), None)                # texto simples (sem tipos especiais do pandas)
gdf["n_ceps"] = gdf["n_ceps"].astype(int)
gdf["score_nome"] = gdf["score_nome"].astype(float)
assert len(gdf) == len(geo), "A camada perdeu feições no merge!"
arq_gpkg = PASTA_SAIDA / "logradouros_status_pareamento.gpkg"
if arq_gpkg.exists():
    arq_gpkg.unlink()                                                               # evita misturar com uma execução anterior
gdf.to_file(arq_gpkg, layer="logradouros_cep", driver="GPKG", engine="pyogrio")
log.info("Camada salva: %s (%d feições, CRS %s)", arq_gpkg.name, len(gdf), gdf.crs)
log.info("Exportação concluída!")

## 10. Resumo da etapa 1
Taxas de pareamento (exato × fuzzy) e volume de resíduos.

In [ ]:
log.info("Resumo da etapa 1...")
nomeados = int((~geo["sem_nome"]).sum())
st = status["status"].value_counts()
log.info("Segmentos do geosanja: %d | nomeados: %d | sem nome: %d (%.1f%%)", len(geo), nomeados, st.get("sem_nome", 0),
         st.get("sem_nome", 0) / len(geo) * 100)
for s in ["pareado", "multi_cep", "ambiguo", "sem_cep"]:
    log.info("  %-10s %6d  (%.1f%% dos nomeados)", s, st.get(s, 0), st.get(s, 0) / nomeados * 100)

por_metodo = (status[status["n_ceps"] > 0]["metodo"].value_counts())
com_cep = int(por_metodo.sum())
exato = int(por_metodo[~por_metodo.index.isin(["fuzzy", "iniciais"])].sum())
iniciais = int(por_metodo.get("iniciais", 0))
fuzzy = int(por_metodo.get("fuzzy", 0))
log.info("Segmentos com CEP: %d (%.1f%% dos nomeados) | exato: %d | iniciais: %d | fuzzy: %d",
         com_cep, com_cep / nomeados * 100, exato, iniciais, fuzzy)
log.info("Por método:\n%s", por_metodo.to_string())

log.info("CEPs distintos nos correios: %d | pareados: %d (%.1f%%) | órfãos: %d",
         correios["cep"].nunique(), len(ceps_pareados), len(ceps_pareados) / correios["cep"].nunique() * 100,
         res_ceps_orfaos["cep"].nunique())
log.info("Arquivos em %s:", PASTA_SAIDA)
for arq in sorted(PASTA_SAIDA.iterdir()):
    log.info("  %s (%.0f KB)", arq.name, arq.stat().st_size / 1024)
log.info("Etapa 02 concluída!")